<a href="https://colab.research.google.com/github/Laiba-saeed92/Deep-Learning-and-NLP-Projects/blob/main/RAG_Application_using__LangChain_GroqLlama_3_1_8b%26_FAISS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
'''This is called retrieval augmented generation (RAG), as you would retrieve the relevant data and use it as augmented context for the LLM.
Instead of relying solely on knowledge derived from the training data,
a RAG workflow pulls relevant information and connects static LLMs with real-time data retrieval.'''

'This is called retrieval augmented generation (RAG), as you would retrieve the relevant data and use it as augmented context for the LLM.\nInstead of relying solely on knowledge derived from the training data,\na RAG workflow pulls relevant information and connects static LLMs with real-time data retrieval.'

In [ ]:
#Install the necessary libraries
!pip install langchain openai tiktoken rapidocr-onnxruntime


In [ ]:
!pip install groq


In [ ]:
from google.colab import userdata #This imports the userdata module from Google Colab.

GROQ_API_KEY = userdata.get('groq_api_key').strip() ##userdata is a way to securely store and retrieve secrets (like API keys) in your Colab environment
#GROQ_API_KEY now contains your API key as a Python variable.

In [ ]:
!pip install langchain-community

In [ ]:
!pip install pypdf

## **Data Ingestion**

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS

In [ ]:
with open("AI.pdf", "rb") as f: #You're opening a PDF file using open(..., "r"), but PDF is binary, not text — so Python cannot decode it as UTF-8.
  data= f.read() #But this will only give you raw binary, not extracted text.

In [ ]:
#TextLoader is for plain UTF-8 text files like .txt, while PyPDFLoader is for PDFs because PDFs are binary and cannot be read with UTF-8.
#UTF-8 works only for real text files, not for binary formats like PDF.
#why pdf are binary
#PDFs are binary because they store more than just text — they contain compressed streams, fonts, images, layout instructions, and graphic objects,
# all of which require binary encoding instead of plain text.
loader= PyPDFLoader("AI.pdf")
document= loader.load()

In [ ]:
print(document[0].page_content)

© 2023 IJRTI | Volume 8, Issue 4 | ISSN: 2456-3315 
  
IJRTI2304061 International Journal for Research Trends and Innovation (www.ijrti.org) 356 
 
RESEARCH PAPER ON ARTIFICIAL INTELLIGENCE & 
ITS APPLICATIONS 
 
 
Prof. Neha Saini 
 
Assistant Professor in Department of Computer Science & IT 
SDAM College Dinanagar 
 
ABSTRACT- 
 
It is the science and engineering of making intelligent machines, especially intelligent computer programs. It is related to 
the similar task of using computers to understand human intelligence, but AI does not have to confine itself to methods that 
are biologically observable. While no consensual definition of Artificial Intelligence (AI) exists, AI is broadly characteriz ed 
as the study of computa tions that allow for perception, reason and action.  Today, the amount of data that is generated, by 
both humans and machines, far outpaces humans’ ability to absorb, interpret, and make complex decisions based on that data. 
Artificial intelligence forms th 

## **Chunking of Data**

In [ ]:
!pip install langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter= RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
text_chunks= text_splitter.split_documents(document)

In [ ]:
text_chunks

[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2023-04-10T10:53:49+05:30', 'author': 'pc 4', 'moddate': '2023-04-10T10:53:49+05:30', 'source': 'AI.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}, page_content='© 2023 IJRTI | Volume 8, Issue 4 | ISSN: 2456-3315 \n  \nIJRTI2304061 International Journal for Research Trends and Innovation (www.ijrti.org) 356 \n \nRESEARCH PAPER ON ARTIFICIAL INTELLIGENCE & \nITS APPLICATIONS \n \n \nProf. Neha Saini \n \nAssistant Professor in Department of Computer Science & IT \nSDAM College Dinanagar \n \nABSTRACT- \n \nIt is the science and engineering of making intelligent machines, especially intelligent computer programs. It is related to'),
 Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2023-04-10T10:53:49+05:30', 'author': 'pc 4', 'moddate': '2023-04-10T10:53:49+05:30', 'source': 'AI.pdf', 'total_pages': 5, 'page': 0, 'pa

In [ ]:
#This is a list of document chunks returned by RecursiveCharacterTextSplitter.
#page_content → the actual text content
#metadata → info like page number, source, etc.
print(text_chunks[0].page_content)

© 2023 IJRTI | Volume 8, Issue 4 | ISSN: 2456-3315 
  
IJRTI2304061 International Journal for Research Trends and Innovation (www.ijrti.org) 356 
 
RESEARCH PAPER ON ARTIFICIAL INTELLIGENCE & 
ITS APPLICATIONS 
 
 
Prof. Neha Saini 
 
Assistant Professor in Department of Computer Science & IT 
SDAM College Dinanagar 
 
ABSTRACT- 
 
It is the science and engineering of making intelligent machines, especially intelligent computer programs. It is related to


In [ ]:
!pip install langchain langchain-community  faiss-cpu




In [ ]:
pip install langchain_huggingface sentence-transformers


In [ ]:
#It converts text → numbers (vectors)but You cannot import embeddings from Groq.Groq is purely for text generation / LLM inference.Embeddings (vectors for FAISS or RAG) must come from a different model
#LangChain changed their package structure.Earlier it was from langchain.embeddings.openai import OpenAIEmbeddings,Now it's moved to a separate package.
#HuggingFace embeddings (if you want an open-source alternative)
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2") #hugging face has this emedding model
#FAISS = vector database created by Facebook AI Research.It stores your embeddings and lets you:Search similar texts,Build RAG systems,Perform fast nearest-neighbor lookup
from langchain_community.vectorstores import FAISS #LangChain moved ALL community integrations (FAISS, Chroma, Pinecone, etc.) into a separate package.

#the chain automatically does something equivalent to:docs = retriever.get_relevant_documents("What is X?")
#under the hood. You don’t see it because the framework abstracts it for you. and bring relevant documents/answer for you,The chain internally knows that context is a retriever.
#The chain “magically” handles embedding the query, searching FAISS, and passing documents to the LLM.
vectorstore=FAISS.from_documents(text_chunks, embeddings)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
retriever= vectorstore.as_retriever()

In [ ]:
from langchain_core.prompts import ChatPromptTemplate


In [ ]:
template="""You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.
query: {query}
Context: {context}
Answer:
"""

In [ ]:
prompt= ChatPromptTemplate.from_template(template) #from_template is a class method, not an instance(object) method.

In [ ]:
!pip install -U langchain-groq

In [ ]:
pip install --upgrade langchain langchain-core


In [ ]:
from langchain_groq import ChatGroq #groq is a subpackage inside langchain framework/package
#from langchain_core.runnables.passthrough import RunnablePassThrough
from langchain_core.output_parsers import StrOutputParser

In [ ]:
class RunnablePassThrough():
  def __call__(self, query):
    return query

In [ ]:
passthrough= RunnablePassThrough()

In [ ]:
from google.colab import userdata
groq_api_key = userdata.get('groq_api_key')

In [ ]:


groq_llm = ChatGroq(
    api_key=groq_api_key,
    model_name="llama-3.1-8b-instant"   # or "llama-3-8b-instant"
)

In [ ]:
output_parser= StrOutputParser()

In [ ]:
#The chain will now take query as input, pass it through passthrough (returns it unchanged),
#feed it into the prompt, then the llm_model, and finally the output_parser.
'''When you run the chain with rag_chain("your question"):
passthrough just returns your query as-is.
The chain knows that "context" is a retriever.
Internally, it calls something like retriever.get_relevant_documents(query) behind the scenes.
Then it passes the retrieved documents to your prompt to generate the answer.'''
rag_chain=({ "context": retriever, "query": passthrough} ## use the object, not class,It takes a query and returns the most relevant document(s) from your vector store
           | prompt            #Converts the inputs (context + query) into a single string prompt that can be fed to the LLM.
           |groq_llm            #Takes the prompt text and generates a response
           | output_parser      #A StrOutputParser that ensures the LLM output is returned as a clean string.
)

In [ ]:
# print(groq_api_key)

In [ ]:
rag_chain.invoke("What is AI?")

'Artificial Intelligence (AI) is the study of computations that allow for perception, reason, and action. It is the branch of computer science that deals with the intelligence of machines, enabling them to perform tasks that make people seem intelligent. The central principles of AI include reasoning, knowledge, planning, learning, communication, perception, and the ability to move and manipulate objects. AI is broadly characterized as the science and engineering of making intelligent machines, especially intelligent computer programs. It does not have to confine itself to methods that are biologically observable, and it is the study of computations that allow for perception, reason, and action. AI enables machines to take actions that maximize their chances of success and is the future of all complex decision-making.'

In [ ]:
rag_chain.invoke("who is the prime minister of pakistan?")

"I don't know who the current Prime Minister of Pakistan is."

In [ ]:
rag_chain.invoke("what is the difference between AI and human intelligence?")

'The main difference between AI and human intelligence lies in their processing power and methods. Human intelligence is biologically observable, whereas AI does not have to confine itself to biological methods. Additionally, AI is not limited by the capacity of the human brain, which is estimated to be around 10 thousand million binary digits. Human intelligence is also limited and volatile, whereas AI can process vast amounts of data and perform tasks efficiently.'